# 🚀 Viettel AI Race 2026 — Inference Validation Notebook
**Mục đích**: Kiểm tra tương thích mô hình LFM2.5-1.2B trên vLLM.

> 💡 **LƯU Ý PHẦN CỨNG TẦM QUAN TRỌNG**:
> - **Tesla T4 (Colab)**: CUDA Capability 7.5 (KHÔNG có phần cứng FP8 Tensor Cores).
> - **MiG H200 (BTC Portal)**: CUDA Capability 9.0 (HỖ TRỢ NATIVE FP8 Tensor Cores).
> - Do đó, cờ `--quantization=fp8` và `--kv-cache-dtype=fp8_e4m3` **SẼ KHÔNG CHẠY ĐƯỢC TRÊN COLAB T4** (vLLM sẽ từ chối khởi động vì T4 thiếu hardware FP8), nhưng **CHẠY HOÀN HẢO TRÊN H200 BTC** (bản v5.0 nộp lên BTC đạt 61.15 điểm thành công).

## Bước 1: Kiểm tra phần cứng GPU & Cài đặt vLLM

In [ ]:
# Xóa triệt để torchaudio & torchvision bị lệch ABI trên Colab
!pip uninstall -y torchaudio torchvision
!rm -rf /usr/local/lib/python*/dist-packages/torchaudio* /usr/local/lib/python*/dist-packages/torchvision*

# Cài vLLM tương thích CUDA 12.x trên Colab T4
!pip install -q 'vllm==0.6.4.post1' aiohttp openai numpy huggingface_hub

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {gpu_name} (CUDA Capability {cap[0]}.{cap[1]})")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    if cap[0] < 8 or (cap[0] == 8 and cap[1] < 9):
        print("\n⚠️ LƯU Ý: GPU hiện tại (Tesla T4) không hỗ trợ FP8 phần cứng.")
        print("   Cờ FP8 chỉ hoạt động trên NVIDIA H200 (Compute Capability 9.0) của BTC!")
else:
    raise RuntimeError("❌ Không có GPU! Vào Runtime → Change runtime type → chọn T4 GPU")

## Bước 2: Tải Model

In [ ]:
import os
from huggingface_hub import snapshot_download

model_dir = "./model"
if not os.path.exists(model_dir):
    print("Downloading LiquidAI/LFM2.5-1.2B-Instruct...")
    snapshot_download(repo_id="LiquidAI/LFM2.5-1.2B-Instruct", local_dir=model_dir)
    print("Done!")
else:
    print("Model already at ./model")

## Bước 3: Khởi chạy vLLM Server trên Colab

Tự động chọn cờ phù hợp với GPU (T4 dùng bfloat16/float16 để test logic, H200 dùng FP8 cho bài nộp chính thức).

In [ ]:
import subprocess, time, requests, torch

gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
supports_fp8 = cap[0] > 8 or (cap[0] == 8 and cap[1] >= 9)

vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model=./model",
    "--served-model-name=LFM2.5-1.2B-Instruct",
    "--host=0.0.0.0",
    "--port=8000",
    "--tensor-parallel-size=1",
    "--max-model-len=4096",
    "--gpu-memory-utilization=0.85",
    "--enable-prefix-caching",
]

if supports_fp8:
    print("✅ GPU hỗ trợ FP8 → Thêm cờ FP8")
    vllm_cmd.extend(["--quantization=fp8", "--kv-cache-dtype=fp8_e4m3"])
else:
    print("ℹ️ GPU Tesla T4 không có FP8 hardware → Chạy mode chuẩn để test API/Accuracy")

print("Command:", ' '.join(vllm_cmd))

!pkill -f "vllm.entrypoints.openai.api_server"
time.sleep(2)

log_file = open("vllm_colab.log", "w")
server_process = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=log_file)
print("\nServer starting... (chờ tối đa 120s)")

ready = False
for i in range(60):
    try:
        resp = requests.get("http://localhost:8000/health", timeout=2)
        if resp.status_code == 200:
            print(f"\n✅ Server sẵn sàng sau {(i+1)*2}s!")
            ready = True
            break
    except:
        pass
    if (i+1) % 5 == 0:
        print(f"  Đang chờ... {(i+1)*2}s")
    time.sleep(2)

if not ready:
    print("\n❌ SERVER KHÔNG KHỞI ĐỘNG ĐƯỢC!")
    log_file.close()
    with open("vllm_colab.log", "r") as f:
        print(f.read()[-2000:])
else:
    print("✅ PASS: Server vLLM khởi động OK!")

## Bước 4: Accuracy Sanity Check

In [ ]:
import requests

GPQA_STYLE_QUESTIONS = [
    {"q": "What is the speed of light in vacuum?\nA) 3x10^8 m/s\nB) 3x10^6 m/s\nC) 3x10^10 m/s\nD) 3x10^4 m/s", "a": "A"},
    {"q": "Which organelle is responsible for ATP synthesis in eukaryotic cells?\nA) Ribosome\nB) Lysosome\nC) Mitochondria\nD) Golgi apparatus", "a": "C"},
    {"q": "What is the derivative of sin(x)?\nA) -cos(x)\nB) cos(x)\nC) tan(x)\nD) -sin(x)", "a": "B"},
    {"q": "In quantum mechanics, what does the Heisenberg Uncertainty Principle state?\nA) Energy is conserved\nB) Position and momentum cannot both be precisely measured\nC) Light travels in straight lines\nD) Entropy always increases", "a": "B"},
    {"q": "Which element has atomic number 79?\nA) Silver\nB) Platinum\nC) Gold\nD) Mercury", "a": "C"},
    {"q": "DNA replication is:\nA) Conservative\nB) Dispersive\nC) Semi-conservative\nD) Random", "a": "C"},
    {"q": "The Pauli exclusion principle states that:\nA) No two fermions can occupy the same quantum state\nB) Energy is quantized\nC) Light has wave-particle duality\nD) Mass-energy equivalence", "a": "A"},
    {"q": "What is the powerhouse of the cell?\nA) Nucleus\nB) Mitochondria\nC) Ribosome\nD) Cell membrane", "a": "B"},
]

def ask_model(question):
    resp = requests.post("http://localhost:8000/v1/chat/completions", json={
        "model": "LFM2.5-1.2B-Instruct",
        "messages": [{"role": "user", "content": f"Answer with ONLY the letter (A, B, C, or D).\n\n{question}"}],
        "max_tokens": 5,
        "temperature": 0.0
    }, timeout=30)
    return resp.json()["choices"][0]["message"]["content"].strip()

correct = 0
print("=== ACCURACY RESULTS ===")
print(f"{'Q':>3} | Expected | Got    | Status")
print("-" * 40)
for i, item in enumerate(GPQA_STYLE_QUESTIONS):
    try:
        answer = ask_model(item["q"])
        ok = item["a"] in answer
        correct += ok
        print(f"{i+1:>3} | {item['a']:^8} | {answer[:6]:^6} | {'✅' if ok else '❌'}")
    except Exception as e:
        print(f"{i+1:>3} | {item['a']:^8} | {'ERROR':^6} | 💥 {e}")

accuracy = correct / len(GPQA_STYLE_QUESTIONS)
print("-" * 40)
print(f"Accuracy: {correct}/{len(GPQA_STYLE_QUESTIONS)} = {accuracy:.1%}")